# SlideOn — sparse inverted-index retrieval on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/SlideOn-sparse-retrieval.ipynb)

[`lightonai/SlideOn`](https://huggingface.co/lightonai/SlideOn) is LightOn's 174M English **sparse** retriever (Apache 2.0). It maps a query or document to one 32,768-dimensional nonnegative sparse vector of **learned latents**, then ranks with a **raw dot product**. Active latents go into an inverted index: the query only walks the posting lists it hits. That is a different retrieval path from the dense EmbeddingGemma 2 notebook (one pooled vector, cosine) and the late-interaction pplx-embed notebook (one vector per token, MaxSim).

The sibling [`lightonai/mSlideOn`](https://huggingface.co/lightonai/mSlideOn) is a 339M multilingual / long-context / code model. This notebook stays on SlideOn so the load stays small.

**Licence:** Apache 2.0. Loading runs custom Hub code (`trust_remote_code=True`).

**Runtime:** pick *Runtime > Change runtime type > T4 GPU* for the fastest run. A CPU runtime also works. The free T4 has 16 GB of VRAM, no native bfloat16, and no FlashAttention 2. This notebook loads the checkpoint in **float16** with **SDPA** (CPU uses float32). The Hub file is 697 MB of float32; float16 weights are about **0.32 GB**. Encoding a handful of short documents stays well under 2 GB.

**Links:** [model card](https://huggingface.co/lightonai/SlideOn) · [blog](https://huggingface.co/blog/lightonai/slideon-mslideon) · [collection](https://huggingface.co/collections/lightonai/slideon-and-mslideon)


In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1"

# The model card requires sentence-transformers>=6.1,<6.2 and transformers>=5.3,<6.
# SparseEncoder, encode_query / encode_document, and this checkpoint's custom
# SlideOn module land in that range. 6.1.0 / 5.20.1 are pinned releases inside it.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version
import sentence_transformers

assert Version(sentence_transformers.__version__) >= Version("6.1.0")
assert Version(transformers.__version__) >= Version("5.3.0")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    # T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    DTYPE = torch.float16
    ATTN = "sdpa"
    print("GPU:", torch.cuda.get_device_name(), f"| VRAM {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    DTYPE = torch.float32
    ATTN = None
    print("CUDA not available — running on CPU. A T4 is faster but not required.")

print("sentence-transformers", sentence_transformers.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("device:", DEVICE, "| dtype:", DTYPE)


## 1. Load SlideOn

`SparseEncoder` plus `trust_remote_code=True` loads the custom SlideOn module from the Hub (`slideon.py`, `modeling_slideon.py`). `encode_query` / `encode_document` add the `query: ` and `document: ` prompts — pass raw text. The first run downloads the 697 MB float32 file; the live weights are about 0.35 GB in float16. The sparse head stays in float32 (`_keep_in_fp32_modules_strict`).


In [ ]:
from sentence_transformers import SparseEncoder

MODEL_ID = "lightonai/SlideOn"


def load_slideon(attn):
    model_kwargs = {"dtype": DTYPE}
    if attn is not None:
        model_kwargs["attn_implementation"] = attn
    return SparseEncoder(
        MODEL_ID,
        trust_remote_code=True,
        device=DEVICE,
        model_kwargs=model_kwargs,
    )


if DEVICE == "cuda":
    try:
        model = load_slideon("sdpa")
        ATTN = "sdpa"
    except Exception as exc:
        print("sdpa load failed, retrying eager:", exc)
        model = load_slideon("eager")
        ATTN = "eager"
else:
    model = load_slideon(None)
    ATTN = "eager"

n_params = sum(p.numel() for p in model.parameters())
bytes_live = sum(p.numel() * p.element_size() for p in model.parameters())
dtypes = sorted({str(p.dtype).replace("torch.", "") for p in model.parameters()})
dim = getattr(model, "get_embedding_dimension", lambda: 32768)() or 32768

print("model:", MODEL_ID)
print("parameters:", f"{n_params / 1e6:.1f}M")
print("live weights:", f"{bytes_live / 1024**3:.2f} GB", f"(dtypes: {', '.join(dtypes)})")
print("fp16 weight estimate:", f"{n_params * 2 / 1024**3:.2f} GB")
print("embedding dim:", dim)
print("similarity fn:", getattr(model, "similarity_fn_name", "dot"))
print("attention:", ATTN, "| device:", getattr(model, "device", DEVICE))
if DEVICE == "cuda":
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 2. Inspect a sparse vector

Each text becomes a 32,768-d nonnegative sparse COO tensor. Dimensions are **learned latents**, not tokenizer vocab ids, so `SparseEncoder.decode()` cannot name them. The card's demo budget is **q-64 / d-512**: keep the top 64 query latents and the top 512 document latents. That cap is Matryoshka pruning — it does not change the vector width.


In [ ]:
queries = ["What causes a solar eclipse?"]
documents = [
    "A solar eclipse occurs when the Moon passes between the Sun and Earth.",
    "Wind turbines convert the kinetic energy of wind into electricity.",
]

query_embs = model.encode_query(queries, max_active_dims=64, show_progress_bar=False)
doc_embs = model.encode_document(documents, max_active_dims=512, show_progress_bar=False)
scores = model.similarity(query_embs, doc_embs)

def nnz(emb):
    emb = emb.detach().cpu()
    if emb.is_sparse:
        return int(emb.coalesce()._nnz())
    return int((emb != 0).sum())


print("query shape:", tuple(query_embs.shape), "| sparse:", bool(query_embs.is_sparse))
print("document shape:", tuple(doc_embs.shape), "| sparse:", bool(doc_embs.is_sparse))
print("query nnz:", nnz(query_embs), "/", query_embs.shape[-1])
print("document nnz:", nnz(doc_embs), "across", doc_embs.shape[0], "docs")
print("dot-product scores:", scores.detach().cpu().tolist())
print()
print("top query latents (id, weight):")
q = query_embs[0].detach().cpu()
if q.is_sparse:
    q = q.coalesce()
    pairs = sorted(zip(q.indices().reshape(-1).tolist(), q.values().tolist()), key=lambda item: item[1], reverse=True)
else:
    nz = torch.nonzero(q.reshape(-1), as_tuple=False).reshape(-1)
    pairs = sorted(((int(i), float(q.reshape(-1)[i])) for i in nz), key=lambda item: item[1], reverse=True)
for latent, weight in pairs[:8]:
    print(f"  {latent:5d}  {weight:.4f}")


## 3. Build a tiny inverted index and rank

For each document, store every nonzero `(latent → weight)` in a posting list. A query walks only the lists for its active latents and accumulates `query_weight * doc_weight`. Documents that share no latents with the query are never scored. On this 12-document set the scores must match `model.similarity` (exact search). The index is the form that scales: a larger corpus still only reads the hit postings.


In [ ]:
from collections import defaultdict

DOCUMENTS = [
    "A solar eclipse occurs when the Moon passes between the Sun and Earth and casts a shadow on the surface.",
    "A lunar eclipse happens when Earth lines up between the Sun and the Moon, and the Moon darkens.",
    "Wind turbines convert the kinetic energy of moving air into electricity for the grid.",
    "If you were charged twice for the same invoice, open Billing and request a refund. Credits post within five business days.",
    "Reset a forgotten password from Settings > Security. The email link expires after thirty minutes.",
    "International orders ship to more than forty countries. Postage is calculated at checkout.",
    "Feed a sourdough starter equal parts flour and water daily and leave the jar at room temperature.",
    "Equity prices dropped after the company missed its quarterly earnings forecast.",
    "A licensed physician can prescribe antibiotics for a bacterial infection, but they do not help a viral cold.",
    "A battery-electric car stores energy in a lithium-ion pack and drives the wheels with an electric motor.",
    "Python's list.sort method orders items in place and returns None.",
    "The checkout page accepts Visa, Mastercard, and American Express.",
]

QUERIES = [
    "What causes a solar eclipse?",
    "Can I get my money back? The payment went through two times.",
    "Who can write a prescription for a bacterial infection?",
    "How does an EV store energy?",
    "Why did share prices drop after the earnings miss?",
]


def iter_sparse_rows(batch):
    # Yield (row, latent_ids, weights) from a [n, dim] sparse COO tensor.
    batch = batch.detach().cpu()
    if batch.is_sparse:
        batch = batch.coalesce()
        if batch.ndim == 1:
            yield 0, batch.indices().reshape(-1).tolist(), batch.values().tolist()
            return
        rows, cols = batch.indices()
        vals = batch.values()
        buckets = defaultdict(lambda: ([], []))
        for row, col, weight in zip(rows.tolist(), cols.tolist(), vals.tolist()):
            buckets[row][0].append(col)
            buckets[row][1].append(weight)
        for row in range(batch.shape[0]):
            ids, weights = buckets[row]
            yield row, ids, weights
        return
    if batch.ndim == 1:
        batch = batch.unsqueeze(0)
    for row, vec in enumerate(batch):
        nz = torch.nonzero(vec, as_tuple=False).reshape(-1)
        yield row, nz.tolist(), vec[nz].tolist()


def build_inverted_index(doc_embs):
    index = defaultdict(list)  # latent -> [(doc_id, weight)]
    nnz = []
    for doc_id, ids, weights in iter_sparse_rows(doc_embs):
        nnz.append(len(ids))
        for latent, weight in zip(ids, weights):
            index[int(latent)].append((doc_id, float(weight)))
    return index, nnz


def search_index(index, query_emb, n_docs):
    scores = [0.0] * n_docs
    lists_hit = 0
    postings = 0
    for _, ids, weights in iter_sparse_rows(query_emb):
        for latent, q_w in zip(ids, weights):
            posting = index.get(int(latent))
            if not posting:
                continue
            lists_hit += 1
            for doc_id, d_w in posting:
                scores[doc_id] += q_w * d_w
                postings += 1
    return scores, lists_hit, postings


doc_embs = model.encode_document(DOCUMENTS, max_active_dims=512, show_progress_bar=False)
query_embs = model.encode_query(QUERIES, max_active_dims=64, show_progress_bar=False)
index, doc_nnz = build_inverted_index(doc_embs)
sim = model.similarity(query_embs, doc_embs).detach().cpu()

n_postings = sum(len(v) for v in index.values())
print(f"index: {len(DOCUMENTS)} docs, {len(index)} unique latents, {n_postings} postings")
print(f"document nnz (d-512): min={min(doc_nnz)} max={max(doc_nnz)} mean={sum(doc_nnz) / len(doc_nnz):.1f}")
print(f"sparsity vs 32,768-d: {sum(doc_nnz) / (len(doc_nnz) * 32768):.4%}\n")

for q_i, query in enumerate(QUERIES):
    scores, lists_hit, n_scored = search_index(index, query_embs[q_i], len(DOCUMENTS))
    ranked = sorted(range(len(DOCUMENTS)), key=lambda i: scores[i], reverse=True)
    sim_top = int(sim[q_i].argmax())
    print(f"query: {query}")
    print(f"  walked {lists_hit} lists / {n_scored} postings | similarity() top-1 matches index: {ranked[0] == sim_top}")
    for rank, doc_id in enumerate(ranked[:3], start=1):
        print(f"  {rank}. {scores[doc_id]:7.3f}  {DOCUMENTS[doc_id]}")
    print()


## 4. Compare to BM25 (and a tiny dense baseline)

Same documents, three rankers:

- **SlideOn** — learned sparse latents, inverted index, raw dot product (q-64 / d-512).
- **BM25** — lexical term overlap, no extra package (Okapi BM25, k1=1.5, b=0.75).
- **MiniLM** — `sentence-transformers/all-MiniLM-L6-v2` (22M, 384-d cosine). Optional dense contrast; skip the MiniLM cell if you only want sparse vs BM25.

This is a toy set, not a BEIR run. Watch the paraphrase queries: BM25 needs shared words; SlideOn and MiniLM can match "EV" to "battery-electric car" or "money back" to "refund".


In [ ]:
import math
import re
from collections import Counter


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25Okapi:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(text) for text in corpus]
        self.n_docs = len(self.docs)
        self.avgdl = sum(map(len, self.docs)) / self.n_docs
        df = Counter()
        for doc in self.docs:
            df.update(set(doc))
        self.idf = {term: math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1.0) for term, n in df.items()}

    def scores(self, query):
        out = []
        for doc in self.docs:
            tf = Counter(doc)
            dl = len(doc)
            score = 0.0
            for term in tokenize(query):
                freq = tf.get(term)
                if not freq:
                    continue
                denom = freq + self.k1 * (1.0 - self.b + self.b * dl / self.avgdl)
                score += self.idf.get(term, 0.0) * (freq * (self.k1 + 1.0)) / denom
            out.append(score)
        return out


def top1(scores):
    return max(range(len(scores)), key=lambda i: scores[i])


bm25 = BM25Okapi(DOCUMENTS)
print(f"{'query':<58}  {'SlideOn':<8} {'BM25':<8}")
print("-" * 80)
for q_i, query in enumerate(QUERIES):
    sparse_scores, _, _ = search_index(index, query_embs[q_i], len(DOCUMENTS))
    lexical = bm25.scores(query)
    s_id, b_id = top1(sparse_scores), top1(lexical)
    print(f"{query[:58]:<58}  {s_id:<8} {b_id:<8}")
    print(f"  SlideOn: {DOCUMENTS[s_id][:90]}")
    print(f"  BM25:    {DOCUMENTS[b_id][:90]}")
    print()


In [ ]:
from sentence_transformers import SentenceTransformer

dense = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
dense_docs = dense.encode(DOCUMENTS, normalize_embeddings=True, show_progress_bar=False)
dense_queries = dense.encode(QUERIES, normalize_embeddings=True, show_progress_bar=False)
dense_scores = dense_queries @ dense_docs.T

print(f"MiniLM parameters: {sum(p.numel() for p in dense.parameters()) / 1e6:.1f}M | dim: {dense.get_embedding_dimension()}")
print()
print(f"{'query':<58}  {'SlideOn':<8} {'BM25':<8} {'MiniLM':<8}")
print("-" * 90)
for q_i, query in enumerate(QUERIES):
    sparse_scores, _, _ = search_index(index, query_embs[q_i], len(DOCUMENTS))
    print(
        f"{query[:58]:<58}  "
        f"{top1(sparse_scores):<8} "
        f"{top1(bm25.scores(query)):<8} "
        f"{int(dense_scores[q_i].argmax()):<8}"
    )


## 5. Matryoshka prune (optional)

Joint query/document Matryoshka training lets you cap `max_active_dims` at inference without retraining. Query caps cut the number of posting lists you open; document caps shorten those lists and the stored weights. The card reports that **q-64 / d-512** stays close to unpruned BEIR-15 (56.66 vs 56.68). Here we only check whether top-1 on this toy set holds as the budget shrinks.


In [ ]:
BUDGETS = [
    ("unpruned", None, None),
    ("q-64 / d-512", 64, 512),
    ("q-16 / d-128", 16, 128),
]


def nnz_row(emb):
    emb = emb.detach().cpu()
    if emb.is_sparse:
        return int(emb.coalesce()._nnz())
    return int((emb != 0).sum())


print(f"{'budget':<16} {'q nnz':<8} {'d nnz (mean)':<14}  top-1 per query")
for name, q_k, d_k in BUDGETS:
    q_vecs = model.encode_query(QUERIES, max_active_dims=q_k, show_progress_bar=False)
    d_vecs = model.encode_document(DOCUMENTS, max_active_dims=d_k, show_progress_bar=False)
    pruned_index, d_nnz = build_inverted_index(d_vecs)
    q_nnz = [nnz_row(q_vecs[i]) for i in range(q_vecs.shape[0])]
    tops = []
    for q_i in range(len(QUERIES)):
        scores, _, _ = search_index(pruned_index, q_vecs[q_i], len(DOCUMENTS))
        tops.append(top1(scores))
    print(f"{name:<16} {sum(q_nnz) / len(q_nnz):<8.1f} {sum(d_nnz) / len(d_nnz):<14.1f}  {tops}")


## Caveats

- **Apache 2.0.** The weights and the custom inference files in the Hub repo are Apache 2.0.
- **`trust_remote_code`.** `modeling_slideon.py` and `slideon.py` execute locally. Pin a revision if you need a frozen snapshot.
- **Sparse vs dense vs late-interaction.** SlideOn emits one sparse 32,768-d vector and scores with a raw dot product over shared latents. That is not a dense encoder (EmbeddingGemma 2, Nemotron Embed, MiniLM) and not late-interaction (pplx-embed). Do not L2-normalize SlideOn outputs.
- **Latents are not words.** Unlike SPLADE, dimensions do not map to tokenizer vocab entries. Inspect indices and weights; `decode()` cannot label them.
- **English only.** For multilingual, long-document, or code retrieval use [`lightonai/mSlideOn`](https://huggingface.co/lightonai/mSlideOn) (339M). This notebook does not load it.
- **Toy corpus.** The rankings above are a walkthrough, not a substitute for BEIR-15 (the card reports 56.68 nDCG@10 unpruned).

## Next steps

- Swap in your own documents and keep the q-64 / d-512 budget, or drop `max_active_dims` for the unpruned vectors.
- Store the same postings in Lucene, OpenSearch, or any inverted-index engine if the corpus grows past an in-memory dict.
- Read the [blog](https://huggingface.co/blog/lightonai/slideon-mslideon) for SAE pretraining, Matryoshka ablations, and the mSlideOn numbers.
